# Product ID 100 - Data Preprocessing Pipeline

## Project Overview

This notebook performs the complete preprocessing and feature engineering workflow for Product ID 100 as part of the PCB Failure Root Cause Analysis project.

The objective of this preprocessing pipeline is to prepare a structured and analysis-ready dataset by integrating:
- Historical PCB transaction data
- Environmental parameters
- Seasonal classifications
- Regional mappings

The generated datasets will later be used for:
- Regional failure analysis
- Seasonal trend analysis
- Environmental correlation studies
- Inventory optimization analysis
- Predictive maintenance and forecasting

---

# Scope of This Notebook

This notebook focuses on the following preprocessing stages:

## 1. Raw SAP Data Cleaning
- Removal of unwanted columns
- Standardization of column names
- Date format conversion
- Filtering year-wise datasets
- Handling missing or inconsistent values

---

## 2. Environmental Data Integration
Historical environmental parameters are merged using:
- SLoc (Storage Location)
- Posting Date

The integrated parameters include:
- Maximum Temperature (Tmax)
- Minimum Temperature (Tmin)
- Relative Humidity (RH)
- Daily Thermal Variation (Delta_T)

Daily thermal variation is calculated using:

ΔT = Tmax - Tmin

This parameter helps estimate thermal stress experienced by PCB systems.

---

## 3. Seasonal Feature Engineering
Additional temporal features are generated including:
- Month
- Season

The data is classified into:
- Winter
- Summer
- Monsoon
- Post-Monsoon

to support seasonal reliability analysis.

---

## 4. Regional Mapping
Each SLoc is mapped to its corresponding operational region to support:
- Region-wise failure analysis
- Environmental trend comparisons
- Inventory distribution studies

---

## 5. Year-wise Dataset Generation
Separate datasets are generated for each year from:
- 2020
- 2021
- 2022
- 2023
- 2024
- 2025
- 2026

For every year, the following datasets are created:

### 1. YEAR_cleaned.csv
Contains cleaned transaction data with standardized dates.

### 2. YEAR_cleaned_temp.csv
Contains cleaned transaction data merged with environmental parameters.

### 3. YEAR_cleaned_temp_season_n_region.csv
Contains:
- Environmental parameters
- Seasonal classification
- Regional mapping

This final dataset is used for further analytics and visualization.

---

# Output of This Notebook

The final outputs generated through this preprocessing pipeline are structured datasets suitable for:
- Exploratory Data Analysis (EDA)
- Statistical correlation analysis
- Visualization dashboards
- Predictive modeling
- Inventory optimization studies

---

# Next Stage

The processed datasets generated in this notebook will be used in:
## 100_analysis.ipynb

for:
- Trend analysis
- Regional analysis
- Seasonal analysis
- Environmental correlation analysis
- Predictive analytics

In [2]:
import sys
!{sys.executable} -m pip install pandas requests geopy


   ---------------------------------------- 0/2 [geographiclib]
   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]




[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
import requests
import time
from geopy.geocoders import Nominatim

In [23]:
cities_df = pd.read_csv("C:/Users/Amey/OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)/Desktop/Amey/Python/cities.csv")

cities_df.head()

,SLOC,City,Location,Region
0,5015,Dadra and Nagar Ha,Dadra and Nagar Ha,East
1,5019,Export HUB,Pune,West 1
2,5061,Kolkata - FSB,Kolkatta,East
3,5062,Ranchi - FSB,Ranchi,East
4,5063,Orissa - FSB,Orissa,East


In [24]:
geolocator = Nominatim(user_agent="pcb_failure_analysis")

location_data = []

for index, row in cities_df.iterrows():

    sloc = row['SLOC']
    city = row['Location']

    # Skip empty locations
    if pd.isna(city) or str(city).strip() == "":

        print(f"Skipping empty location for SLOC: {sloc}")

        continue

    try:

        print(f"Fetching coordinates for {city}...")

        location = geolocator.geocode(f"{city}, India")

        if location:

            location_data.append({
                'SLOC': sloc,
                'Location': city,
                'Latitude': location.latitude,
                'Longitude': location.longitude
            })

            print("Done")

        else:

            print(f"Coordinates not found for {city}")

    except Exception as e:

        print(f"Error for {city}: {e}")

    time.sleep(1)

Fetching coordinates for Dadra and Nagar Ha...
Done
Fetching coordinates for Pune...
Done
Fetching coordinates for Kolkatta...
Done
Fetching coordinates for Ranchi...
Done
Fetching coordinates for Orissa...
Done
Fetching coordinates for Guwahati...
Done
Fetching coordinates for Patna...
Done
Fetching coordinates for Nagaland...
Done
Fetching coordinates for Arunachal Pradesh...
Done
Fetching coordinates for Meghalaya...
Done
Fetching coordinates for Siliguri...
Done
Fetching coordinates for Sikkim...
Done
Fetching coordinates for Mizoram...
Done
Fetching coordinates for Pune...
Done
Fetching coordinates for Aurangabad...
Done
Fetching coordinates for Delhi...
Done
Fetching coordinates for Uttar Pradesh...
Done
Fetching coordinates for Haryana...
Done
Fetching coordinates for Jaipur...
Done
Fetching coordinates for Uttaranchal...
Done
Fetching coordinates for Haryana...
Done
Fetching coordinates for Delhi...
Done
Fetching coordinates for Gwalior...
Done
Fetching coordinates for Lucknow.

In [ ]:
locations_df = pd.DataFrame(location_data)

print(locations_df.head())



# ============================================================
# SAVE LOCATION DATASET
# ============================================================

locations_df.to_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\locations.csv",
    index=False
)

print("locations.csv saved successfully")


   SLOC            Location   Latitude  Longitude
0  5015  Dadra and Nagar Ha  20.233792  73.049517
1  5019                Pune  18.521374  73.854507
2  5061            Kolkatta  12.957932  77.641505
3  5062              Ranchi  23.370050  85.325039
4  5063              Orissa  20.543124  84.689732
locations.csv saved successfully


In [26]:
all_data = []

START_DATE = "20200101"
END_DATE = "20261231"

for index, row in locations_df.iterrows():

    sloc = row['SLOC']
    location = row['Location']
    latitude = row['Latitude']
    longitude = row['Longitude']

    print(f"Fetching weather data for {location}...")

    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters=T2M,T2M_MAX,T2M_MIN,RH2M"
        f"&community=RE"
        f"&longitude={longitude}"
        f"&latitude={latitude}"
        f"&start={START_DATE}"
        f"&end={END_DATE}"
        f"&format=JSON"
    )

    response = requests.get(url)

    data = response.json()

    parameters = data['properties']['parameter']

    tavg = parameters['T2M']
    tmax = parameters['T2M_MAX']
    tmin = parameters['T2M_MIN']
    rh = parameters['RH2M']

    for date in tavg.keys():

        all_data.append({

            'SLOC': sloc,
            'Location': location,
            'Date': date,
            'Tavg': tavg[date],
            'Tmax': tmax[date],
            'Tmin': tmin[date],
            'RH': rh[date]

        })

    time.sleep(1)
temperature_df = pd.DataFrame(all_data)

print(temperature_df.head())
temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    format='%Y%m%d'
)
temperature_df['Month'] = temperature_df['Date'].dt.month

temperature_df['Year'] = temperature_df['Date'].dt.year
def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5, 6]:
        return 'Summer'

    elif month in [7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'


temperature_df['Season'] = temperature_df['Month'].apply(get_season)
temperature_df['Delta_T'] = (
    temperature_df['Tmax'] -
    temperature_df['Tmin']
)
print(temperature_df.isnull().sum())
temperature_df.to_csv(
    r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv",
    index=False
)

print("clean_temperature_data.csv saved successfully")
print(temperature_df.head())

Fetching weather data for Dadra and Nagar Ha...
Fetching weather data for Pune...
Fetching weather data for Kolkatta...
Fetching weather data for Ranchi...
Fetching weather data for Orissa...
Fetching weather data for Guwahati...
Fetching weather data for Patna...
Fetching weather data for Nagaland...
Fetching weather data for Arunachal Pradesh...
Fetching weather data for Meghalaya...
Fetching weather data for Siliguri...
Fetching weather data for Sikkim...
Fetching weather data for Mizoram...
Fetching weather data for Pune...
Fetching weather data for Aurangabad...
Fetching weather data for Delhi...
Fetching weather data for Uttar Pradesh...
Fetching weather data for Haryana...
Fetching weather data for Jaipur...
Fetching weather data for Uttaranchal...
Fetching weather data for Haryana...
Fetching weather data for Delhi...
Fetching weather data for Gwalior...
Fetching weather data for Lucknow...
Fetching weather data for Udaipur...
Fetching weather data for Agra...
Fetching weather 

In [ ]:
import pandas as pd


main_file = pd.read_csv("100.csv")

lookup_file = pd.read_csv(r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\Equipment_No_and_Technician_name.csv")

main_file.columns = main_file.columns.str.strip()
lookup_file.columns = lookup_file.columns.str.strip()

duplicates = lookup_file[
    lookup_file.duplicated(subset="Order", keep=False)
]

if not duplicates.empty:
    duplicates.to_csv("duplicates.csv", index=False)
    print("duplicates.csv created")
else:
    print("No duplicate orders found")

lookup_clean = lookup_file.drop_duplicates(
    subset="Order",
    keep="first"
)# Convert order columns to string
main_file["order"] = main_file["Order"].astype(str).str.strip()
lookup_clean["Order"] = lookup_clean["Order"].astype(str).str.strip()

merged = main_file.merge(
    lookup_clean[["Order", "Equipment", "Technician name"]],
    left_on="Order",
    right_on="Order",
    how="left"
)

missing = merged[
    merged["Equipment"].isna()
]

if not missing.empty:
    missing.to_csv("missing.csv", index=False)
    print("missing.csv created")
else:
    print("No missing orders found")

merged.drop(columns=["Order"], inplace=True)

merged.to_csv("100_new.csv", index=False)

print("100_new.csv created successfully")

No duplicate orders found
missing.csv created
100_new.csv created successfully


In [34]:
import pandas as pd
import os

# ============================================================
# DEFINE PATHS
# ============================================================

base_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100"

temperature_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\clean_temperature_data.csv"

cities_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\cities.csv"

# ============================================================
# READ MAIN PCB DATA
# ============================================================

df = pd.read_csv(
    os.path.join(base_path, "100_new.csv")
)

# ============================================================
# REMOVE UNNAMED COLUMNS
# ============================================================

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()
# ============================================================
# REMOVE ROWS WITH MISSING EQUIPMENT OR TECHNICIAN
# ============================================================

missing_equipment_technician = df[

    (
        df['Equipment'].isnull()
    )

    |

    (
        df['Technician name'].isnull()
    )

    |

    (
        df['Equipment']
        .astype(str)
        .str.strip() == ""
    )

    |

    (
        df['Technician name']
        .astype(str)
        .str.strip() == ""
    )

]

# ============================================================
# SAVE MISSING ROWS
# ============================================================

missing_equipment_technician.to_csv(

    os.path.join(
        base_path,
        "missing.csv"
    ),

    index=False

)

print("missing.csv saved successfully")

# ============================================================
# REMOVE MISSING ROWS FROM MAIN DATASET
# ============================================================

df = df[

    ~(
        (
            df['Equipment'].isnull()
        )

        |

        (
            df['Technician name'].isnull()
        )

        |

        (
            df['Equipment']
            .astype(str)
            .str.strip() == ""
        )

        |

        (
            df['Technician name']
            .astype(str)
            .str.strip() == ""
        )
    )

]

# ============================================================
# CONVERT POSTING DATE
# ============================================================

df['Pstng Date'] = pd.to_datetime(
    df['Pstng Date'],
    errors='coerce'
)

# ============================================================
# REMOVE INVALID DATES
# ============================================================

df = df[df['Pstng Date'].notnull()]

# ============================================================
# READ TEMPERATURE DATA
# ============================================================

temperature_df = pd.read_csv(
    temperature_path
)

temperature_df.columns = (
    temperature_df.columns.str.strip()
)

temperature_df['Date'] = pd.to_datetime(
    temperature_df['Date'],
    errors='coerce'
)

# ============================================================
# READ CITIES DATA
# ============================================================

cities_df = pd.read_csv(
    cities_path
)

cities_df.columns = (
    cities_df.columns.str.strip()
)

# ============================================================
# STANDARDIZE SLOC AS STRING
# ============================================================

df['SLoc'] = (
    df['SLoc']
    .astype(str)
    .str.strip()
)

temperature_df['SLOC'] = (
    temperature_df['SLOC']
    .astype(str)
    .str.strip()
)

cities_df['SLOC'] = (
    cities_df['SLOC']
    .astype(str)
    .str.strip()
)

# ============================================================
# REMOVE ONLY TRUE INVALID SLOC VALUES
# ============================================================

invalid_rows = df[

    df['SLoc'].isnull()

    |

    (
        df['SLoc']
        .astype(str)
        .str.strip() == ""
    )

]

# ============================================================
# SAVE INVALID ROWS
# ============================================================

invalid_rows.to_csv(

    os.path.join(
        base_path,
        "invalid.csv"
    ),

    index=False

)

print("invalid.csv saved successfully")

# ============================================================
# REMOVE INVALID ROWS
# ============================================================

df = df[

    ~(
        df['SLoc'].isnull()

        |

        (
            df['SLoc']
            .astype(str)
            .str.strip() == ""
        )
    )

]

# ============================================================
# STANDARDIZE DATE FORMAT
# ============================================================

df['Merge_Date'] = pd.to_datetime(

    df['Pstng Date'],
    errors='coerce'

).dt.normalize()

temperature_df['Merge_Date'] = pd.to_datetime(

    temperature_df['Date'],
    errors='coerce'

).dt.normalize()

# ============================================================
# CREATE YEAR COLUMN
# ============================================================

df['Year'] = df['Pstng Date'].dt.year

# ============================================================
# PROCESS YEARWISE
# ============================================================

for year in range(2020, 2027):

    print("\n================================================")
    print(f"Processing Year: {year}")
    print("================================================")

    # ========================================================
    # CREATE YEAR FOLDER
    # ========================================================

    year_folder = os.path.join(
        base_path,
        str(year)
    )

    os.makedirs(
        year_folder,
        exist_ok=True
    )

    # ========================================================
    # FILTER YEAR DATA
    # ========================================================

    year_df = df[
        df['Year'] == year
    ].copy()

    # ========================================================
    # SAVE CLEANED DATA
    # ========================================================

    cleaned_path = os.path.join(
        year_folder,
        f"{year}_cleaned.csv"
    )

    year_df.to_csv(
        cleaned_path,
        index=False
    )

    print(f"{year}_cleaned.csv saved")

    # ========================================================
    # MERGE TEMPERATURE DATA
    # ========================================================

    merged_df = pd.merge(

        year_df,

        temperature_df[[
            'SLOC',
            'Merge_Date',
            'Tavg',
            'Tmax',
            'Tmin',
            'RH',
            'Month',
            'Season',
            'Delta_T'
        ]],

        left_on=[
            'SLoc',
            'Merge_Date'
        ],

        right_on=[
            'SLOC',
            'Merge_Date'
        ],

        how='left'

    )

    # ========================================================
    # DEBUG UNMATCHED TEMPERATURE ROWS
    # ========================================================

    unmatched_temp = merged_df[
        merged_df['Tmax'].isnull()
    ]

    unmatched_temp.to_csv(

        os.path.join(
            year_folder,
            "unmatched_temperature_rows.csv"
        ),

        index=False

    )

    print(
        f"Unmatched temperature rows: "
        f"{len(unmatched_temp)}"
    )

    # ========================================================
    # MERGE REGION DATA
    # ========================================================

    merged_df = pd.merge(

        merged_df,

        cities_df[[
            'SLOC',
            'Region',
            'Location'
        ]],

        left_on='SLoc',
        right_on='SLOC',

        how='left'

    )

    # ========================================================
    # DEBUG UNMATCHED REGION ROWS
    # ========================================================

    unmatched_region = merged_df[
        merged_df['Region'].isnull()
    ]

    unmatched_region.to_csv(

        os.path.join(
            year_folder,
            "unmatched_region_rows.csv"
        ),

        index=False

    )

    print(
        f"Unmatched region rows: "
        f"{len(unmatched_region)}"
    )

    # ========================================================
    # DROP EXTRA COLUMNS
    # ========================================================

    merged_df.drop(

        columns=[
            'SLOC_x',
            'SLOC_y',
            'Merge_Date'
        ],

        errors='ignore',
        inplace=True

    )

    # ========================================================
    # SAVE INTERMEDIATE FILE
    # ========================================================

    temp_path = os.path.join(
        year_folder,
        f"{year}_cleaned_temp.csv"
    )

    merged_df.to_csv(
        temp_path,
        index=False
    )

    print(f"{year}_cleaned_temp.csv saved")

    # ========================================================
    # SAVE FINAL YEAR FILE
    # ========================================================

    final_path = os.path.join(
        year_folder,
        f"{year}_cleaned_temp_season_n_region.csv"
    )

    merged_df.to_csv(
        final_path,
        index=False
    )

    print(
        f"{year}_cleaned_temp_season_n_region.csv saved"
    )

    # ========================================================
    # NULL CHECK SUMMARY
    # ========================================================

    print("\nNull Summary:")

    print(

        merged_df[
            [
                'Tmax',
                'RH',
                'Season',
                'Region'
            ]
        ].isnull().sum()

    )

# ============================================================
# COMBINE ALL YEARS
# ============================================================

all_years = []

for year in range(2020, 2027):

    final_file = os.path.join(

        base_path,
        str(year),
        f"{year}_cleaned_temp_season_n_region.csv"

    )

    temp_df = pd.read_csv(
        final_file
    )

    all_years.append(
        temp_df
    )

# ============================================================
# CREATE FINAL COMBINED DATASET
# ============================================================

final_combined_df = pd.concat(

    all_years,
    ignore_index=True

)

# ============================================================
# SAVE FINAL EXCEL FILE
# ============================================================

final_output_path = os.path.join(

    base_path,
    "100_Pre_done_Combined.xlsx"

)

final_combined_df.to_excel(

    final_output_path,
    index=False

)

# ============================================================
# SUCCESS MESSAGE
# ============================================================

print("\n================================================")
print("100_Pre_done_Combined.xlsx saved successfully")
print("================================================")

print("\nFinal Shape:")
print(final_combined_df.shape)

print("\nPreview:")
print(final_combined_df.head())

missing.csv saved successfully
invalid.csv saved successfully

Processing Year: 2020
2020_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2020_cleaned_temp.csv saved
2020_cleaned_temp_season_n_region.csv saved

Null Summary:
Tmax      0
RH        0
Season    0
Region    0
dtype: int64

Processing Year: 2021
2021_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2021_cleaned_temp.csv saved
2021_cleaned_temp_season_n_region.csv saved

Null Summary:
Tmax      0
RH        0
Season    0
Region    0
dtype: int64

Processing Year: 2022
2022_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2022_cleaned_temp.csv saved
2022_cleaned_temp_season_n_region.csv saved

Null Summary:
Tmax      0
RH        0
Season    0
Region    0
dtype: int64

Processing Year: 2023
2023_cleaned.csv saved
Unmatched temperature rows: 0
Unmatched region rows: 0
2023_cleaned_temp.csv saved
2023_cleaned_temp_season_n_region.csv saved

Null Summary:
Tma

In [4]:
import pandas as pd
import os

# =========================
# INPUT FILE PATH
# =========================
input_file = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\Preprocessing\100_Pre_done_Combined.xlsx"

# =========================
# OUTPUT FOLDER + FILE NAME
# =========================
output_folder = "Preprocessing"
output_file = "100_2022_26_month_wise.xlsx"

# Create folder if not present
os.makedirs(output_folder, exist_ok=True)

# Full output path
output_path = os.path.join(output_folder, output_file)

# =========================
# READ EXCEL FILE
# =========================
xls = pd.ExcelFile(input_file)

# Read first sheet
df = pd.read_excel(input_file, sheet_name=xls.sheet_names[0])

# =========================
# CONVERT DATE COLUMN
# =========================
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'], errors='coerce')

# =========================
# FILTER DATA FROM 2022 TO 2026
# =========================
df_filtered = df[
    (df['Pstng Date'].dt.year >= 2022) &
    (df['Pstng Date'].dt.year <= 2026)
].copy()

# =========================
# CREATE ABSOLUTE QUANTITY COLUMN
# =========================
df_filtered['Absolute_Quantity'] = df_filtered['Quantity'].abs()

# =========================
# CREATE MONTH COLUMN
# =========================
df_filtered['Month'] = df_filtered['Pstng Date'].dt.to_period('M').astype(str)

# =========================
# MONTHLY CONSUMPTION ANALYSIS
# =========================
monthly_consumption = (
    df_filtered.groupby('Month')['Absolute_Quantity']
    .sum()
    .reset_index()
)

monthly_consumption.rename(
    columns={'Absolute_Quantity': 'Monthly_Consumption'},
    inplace=True
)

# =========================
# SAVE TO EXCEL
# =========================
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1 -> Complete filtered data (2022-2026)
    df_filtered.to_excel(
        writer,
        sheet_name='2022_2026_Data',
        index=False
    )

    # Sheet 2 -> Monthly Consumption
    monthly_consumption.to_excel(
        writer,
        sheet_name='Monthly_Consumption',
        index=False
    )

print(f"\nFile saved successfully at:\n{output_path}")


File saved successfully at:
Preprocessing\100_2022_26_month_wise.xlsx


In [5]:
import pandas as pd
import os

# =========================
# INPUT FILE PATH
# =========================
input_file = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\Preprocessing\100_Pre_done_Combined.xlsx"

# =========================
# OUTPUT FOLDER + FILE NAME
# =========================
output_folder = "Preprocessing"
output_file = "100_2020_21_month_wise.xlsx"

# Create folder if not present
os.makedirs(output_folder, exist_ok=True)

# Full output path
output_path = os.path.join(output_folder, output_file)

# =========================
# READ EXCEL FILE
# =========================
xls = pd.ExcelFile(input_file)

# Read first sheet
df = pd.read_excel(input_file, sheet_name=xls.sheet_names[0])

# =========================
# CONVERT DATE COLUMN
# =========================
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'], errors='coerce')

# =========================
# FILTER DATA FROM 2020 TO 2021
# =========================
df_filtered = df[
    (df['Pstng Date'].dt.year >= 2020) &
    (df['Pstng Date'].dt.year <= 2021)
].copy()

# =========================
# CREATE ABSOLUTE QUANTITY COLUMN
# =========================
df_filtered['Absolute_Quantity'] = df_filtered['Quantity'].abs()

# =========================
# CREATE MONTH COLUMN
# =========================
df_filtered['Month'] = df_filtered['Pstng Date'].dt.to_period('M').astype(str)

# =========================
# MONTHLY CONSUMPTION ANALYSIS
# =========================
monthly_consumption = (
    df_filtered.groupby('Month')['Absolute_Quantity']
    .sum()
    .reset_index()
)

monthly_consumption.rename(
    columns={'Absolute_Quantity': 'Monthly_Consumption'},
    inplace=True
)

# =========================
# SAVE TO EXCEL
# =========================
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1 -> Complete filtered data (2020-2021)
    df_filtered.to_excel(
        writer,
        sheet_name='2020_2021_Data',
        index=False
    )

    # Sheet 2 -> Monthly Consumption
    monthly_consumption.to_excel(
        writer,
        sheet_name='Monthly_Consumption',
        index=False
    )

print(f"\nFile saved successfully at:\n{output_path}")


File saved successfully at:
Preprocessing\100_2020_21_month_wise.xlsx
